In [0]:
%sql
SHOW EXTERNAL LOCATIONS

In [0]:
filename = dbutils.widgets.get('filename')
fnamewithoutext = filename.split('.')[0]

In [0]:
orders_df = spark.read.csv('abfss://sales@sahhe3.dfs.core.windows.net/landing/{}'.format(filename), inferSchema = True, header = True)
display(orders_df.limit(5))

In [0]:
errorFlg = False
orders_count = orders_df.count()
print(orders_count)

distinct_orderid_count = orders_df.select("order_id").distinct().count()
print(distinct_orderid_count)

In [0]:
if orders_count != distinct_orderid_count:
  errorFlg = True
if errorFlg:
    dbutils.fs.mv('abfss://sales@sahhe3.dfs.core.windows.net/landing/{}'.format(filename), 'abfss://sales@sahhe3.dfs.core.windows.net/discarded')
    dbutils.notebook.exit('{"errorFlg": "true", "errorMsg": "Duplicate order_id found"}')

orders_df.createOrReplaceTempView('orders')

In [0]:
dbServer = 'hhe3sqlserver'
dbPort = '1433'
dbName = 'sqldbhhe3'
dbUser = 'hhe3'
dbPassword = 'SQLpassword1' #This is stored in the Azure key vault. Will be retrieved in cell 8 as a secret.
databricksScope = 'sales_scope'

In [0]:
# List existing secrets in the scope
secrets = dbutils.secrets.list(scope='sales_scope')
for s in secrets:
    print(s)

In [0]:
dbPassword = dbutils.secrets.get(scope = databricksScope, key = 'SQLServerLogin')
print(dbPassword)

In [0]:
connection_url = 'jdbc:sqlserver://{}.database.windows.net:{}; database={}; user={}'.format(dbServer, dbPort, dbName, dbUser)

connection_properties = {
  'password': dbPassword,
  'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'}

In [0]:
valid_order_status = spark.read.jdbc(url = connection_url, table = 'dbo.valid_order_status', properties = connection_properties)
display(valid_order_status)

In [0]:
valid_order_status.createOrReplaceTempView('valid_order_status')
invalid_rows = spark.sql('SELECT * FROM orders WHERE order_status NOT IN (SELECT * FROM valid_order_status)')
display(invalid_rows)

In [0]:
if invalid_rows.count() > 0:
  errorFlg = True
if errorFlg:
    dbutils.fs.mv('abfss://sales@sahhe3.dfs.core.windows.net/landing/{}'.format(filename), 'abfss://sales@sahhe3.dfs.core.windows.net/discarded')
    dbutils.notebook.exit('{"errorFlg": "true", "errorMsg": "Invalid order status found"}')
else:
    dbutils.fs.mv('abfss://sales@sahhe3.dfs.core.windows.net/landing/{}'.format(filename), 'abfss://sales@sahhe3.dfs.core.windows.net/staging')
    dbutils.notebook.exit('{"errorFlg": "false", "errorMsg": "No invalid order status found"}')

In [0]:
orders_items_df = spark.read.csv('abfss://sales@sahhe3.dfs.core.windows.net/order_items/order_items.csv', inferSchema = True, header = True)
orders_items_df.createOrReplaceTempView('orders_items')
display(orders_items_df.limit(5))

In [0]:
customers_df = spark.read.jdbc(url = connection_url, table = 'dbo.customers', properties = connection_properties)
customers_df.createOrReplaceTempView('customers')
display(customers_df.limit(5))

In [0]:
orders_df = spark.read.csv('abfss://sales@sahhe3.dfs.core.windows.net/staging/{}'.format(filename), inferSchema = True, header = True)
orders_df.createOrReplaceTempView('orders')

In [0]:
result_df = spark.sql("""
          SELECT customers.customer_id, customers.customer_fname, customers.customer_lname, customers.customer_city, customers.customer_state, customers.customer_zipcode, 
          COUNT(order_id) AS num_orders_placed, 
          ROUND(SUM(order_item_subtotal),2) AS total_amount 
          FROM customers, orders, orders_items 
          WHERE customers.customer_id = orders.customer_id 
          AND orders.order_id = orders_items.order_item_order_id 
          GROUP BY customers.customer_id, customers.customer_fname, customers.customer_lname, customers.customer_city, customers.customer_state, customers.customer_zipcode 
          ORDER BY total_amount DESC
          """)
          
display(result_df.limit(5))

In [0]:
result_df.write.jdbc(url = connection_url, table = 'dbo.sales_reporting', mode = 'overwrite', properties = connection_properties)